In [2]:
import pandas as pd
import psycopg2
from datetime import datetime

import pandas as pd
import psycopg2

# Configura tus datos de conexión
DB_HOST = '69.48.206.219'
DB_PORT = '5432'
DB_NAME = 'collection_db'
DB_USER = 'cobranza'
DB_PASS = 'cobranza2025'

# Leer el CSV
df = pd.read_csv('../data/debtors.csv')

# Limpieza y conversión de datos
def clean_value(val):
    if pd.isnull(val) or str(val).strip() == "0":
        return None
    return str(val).strip()

def parse_date(val):
    if pd.isnull(val) or str(val).strip() == "0":
        return None
    try:
        return datetime.strptime(val, "%m/%d/%Y").date()
    except:
        try:
            return datetime.strptime(val, "%Y-%m-%d").date()
        except:
            return None

def map_marital_status(value):
    mapping = {
        "CASADO(A)": "Casado",
        "CASADO": "Casado",
        "SOLTERO(A)": "Soltero",
        "SOLTERO": "Soltero",
        "UNION LIBRE": "Unión Libre",
        "UNIÓN LIBRE": "Unión Libre",
        "DIVORCIADO(A)": "Separado",
        "DIVORCIADO": "Separado",
        "SEPARADO(A)": "Separado",
        "SEPARADO": "Separado",
        "0": None,
        "": None,
        None: None
    }
    return mapping.get(str(value).strip().upper(), None)

# Conexión a PostgreSQL
conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASS
)
cur = conn.cursor()

for _, row in df.iterrows():
    # --- Insertar en debtors ---
    has_ine = clean_value(row['RFC_Deudor']) is not None
    economic_dependents = int(row['Dependientes']) if clean_value(row['Dependientes']) is not None else 0

    cur.execute("""
        INSERT INTO collection.debtors
        (name, last_name, second_last_name, birth_date, birth_place, gender, marital_status, has_ine, economic_dependents, company_name, occupation)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        RETURNING id
    """, (
        clean_value(row['Nombre_Deudor']),
        clean_value(row['Apellido1_Deudor']),
        clean_value(row['Apellido2_Deudor']),
        parse_date(row['FechaNacimiento']),
        clean_value(row['EntidadNacimiento']),
        clean_value(row['Genero_Deudor']),
        map_marital_status(row['EstadoCivil_Deudor']),
        has_ine,
        economic_dependents,
        clean_value(row['Ultima_Ocupacion']),
        clean_value(row['Ultimo_Puesto'])
    ))
    result = cur.fetchone()
    if result is None:
        raise Exception("Failed to insert debtor or fetch id.")
    debtor_id = result[0]

    # --- Insertar identificaciones ---
    ident_types = [
        ('RFC', clean_value(row['RFC_Deudor']), True if clean_value(row['RFC_Deudor']) else False),
        ('CURP', clean_value(row['CURP_Deudor']), False),
        ('NSS', clean_value(row['NSS_Deudor']), False)
    ]
    for ident_type, ident_number, is_primary in ident_types:
        if ident_number:
            cur.execute("""
                INSERT INTO collection.debtor_identifications
                (debtor_id, identification_type, identification_number, is_primary)
                VALUES (%s, %s, %s, %s)
            """, (
                debtor_id,
                ident_type,
                ident_number,
                is_primary
            ))

    # --- Insertar referencia de cónyuge ---
    spouse = clean_value(row['NombreConyuge'])
    if spouse:
        cur.execute("""
            INSERT INTO collection.debtor_references
            (debtor_id, name, relationship, phone, status)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            debtor_id,
            spouse,
            'Spouse',
            '',  # No hay teléfono en el archivo
            'New'
        ))

conn.commit()
cur.close()
conn.close()
print("Datos insertados correctamente.")

Datos insertados correctamente.
